# Phase 2: Exploratory Data Analysis

Customer, churn, and revenue insights for telecom retention strategy.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config import DATA_PROCESSED, IMAGES_DIR
from src.features import add_engineered_features

In [ ]:
df = add_engineered_features(pd.read_csv(DATA_PROCESSED))
total = len(df)
churned = df["churn_flag"].sum()
active = total - churned
print(f"Total customers: {total:,}")
print(f"Active customers: {active:,}")
print(f"Churned customers: {churned:,}")
print(f"Churn rate: {df['churn_flag'].mean() * 100:.2f}%")

In [ ]:
segment_cols = ["Contract", "PaymentMethod", "tenure_bucket", "InternetService"]
for col in segment_cols:
    summary = (
        df.groupby(col)["churn_flag"]
        .agg(customers="count", churn_rate=lambda s: round(s.mean() * 100, 2))
        .sort_values("churn_rate", ascending=False)
    )
    print(f"\nChurn by {col}")
    display(summary)

In [ ]:
monthly_revenue = df.loc[~df["churn_flag"], "MonthlyCharges"].sum()
revenue_lost = df.loc[df["churn_flag"], "MonthlyCharges"].sum()
avg_clv = df["estimated_clv"].mean()
print(f"Monthly revenue (active): ${monthly_revenue:,.2f}")
print(f"Revenue lost to churn: ${revenue_lost:,.2f}")
print(f"Average estimated CLV: ${avg_clv:,.2f}")

In [ ]:
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="Churn", palette=["#2E86AB", "#E94F37"], ax=axes[0])
axes[0].set_title("Churn Distribution")

contract_rate = df.groupby("Contract")["churn_flag"].mean().sort_values(ascending=False) * 100
sns.barplot(x=contract_rate.index, y=contract_rate.values, palette="Reds_r", ax=axes[1])
axes[1].set_title("Churn Rate by Contract")
axes[1].set_ylabel("Churn Rate (%)")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "eda_overview.png", dpi=150)
plt.show()

## Business observations

- Month-to-month contracts show the highest churn risk.
- Customers in the first year of tenure churn disproportionately.
- Electronic check payment users are a high-risk segment.
- Fiber optic internet customers churn more than DSL, suggesting price/value sensitivity.